# Veriyi Okuma ve Birleştirme

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, concat_ws, lower, regexp_replace

# Spark oturumunu büyük veri için optimize ederek başlatalım
spark = SparkSession.builder \
    .appName("AmazonBigDataProcessing") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

# Dosya yolunu tanımlayalım
data_path = "/home/jovyan/.cache/kagglehub/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9"

# Tüm .tsv dosyalarını tek bir devasa tablo olarak okuyoruz
# Not: Amazon verileri sekmeyle (\t) ayrılır.
df_raw = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .csv(f"{data_path}/*.tsv")

print(f"Başarıyla okunan toplam satır sayısı: {df_raw.count()}")

Başarıyla okunan toplam satır sayısı: 109830520


# Etiketleme ve Metin Temizliği

In [2]:
# 1. Gerekli sütunları seç (Hafıza tasarrufu için sadece 3 sütun: "star_rating", "review_headline", "review_body")
# 2. 1-2 yıldız -> 0 (Kötü), 3 yıldız -> 1 (Nötr), 4-5 yıldız -> 2 (İyi)
# 3. Başlık ve yorumu birleştirip temizle
df_processed = df_raw.select("star_rating", "review_headline", "review_body") \
    .withColumn("label", 
        when(col("star_rating") <= 2, 0)
        .when(col("star_rating") == 3, 1)
        .otherwise(2)) \
    .withColumn("text", lower(concat_ws(" ", col("review_headline"), col("review_body")))) \
    .withColumn("text", regexp_replace(col("text"), "[^a-zA-Z\\s]", "")) \
    .select("label", "text")

# "Mühürleme" (Parquet Olarak Kaydetme)

In [3]:
# Bu işlem verinin büyüklüğüne göre biraz zaman alacaktır.
# Kaydedildiğinde /home/jovyan/data/ klasöründe kalıcı olacak.
df_processed.write.mode("overwrite").parquet("/home/jovyan/data/amazon_final_processed.parquet")

print("Veri başarıyla etiketlendi ve Parquet formatına dönüştürüldü!")

Veri başarıyla etiketlendi ve Parquet formatına dönüştürüldü!
